# Due strati senza niente in mezzo sono uno strato solo

*Anatomia di un LLM* · capitolo 3, I mattoni · script `01-04`, `scripts/01_04_non_linearita.py`

E' la ragione per cui esistono le funzioni di attivazione, e si dimostra in
mezza pagina: si prendono due strati lineari, si calcola la matrice che fa
esattamente lo stesso lavoro, e si verifica che coincidano. Poi si mette una
funzione in mezzo e quella matrice smette di esistere.

Si esegue con `python3 scripts/01_04_non_linearita.py`
e dura circa 5 secondi.

> **Per riottenere i numeri del libro** esegui le celle in ordine, dalla prima. Il seme si fissa una volta sola, in `prepara()`: rieseguire una cella a metà cambia i numeri casuali di tutte quelle dopo.

[Apri in Colab](https://colab.research.google.com/github/fcollova/llm-anatomy/blob/main/notebooks/01_04_non_linearita.ipynb) · la prima cella scarica il codice, se non lo trova accanto.

In [1]:
# Porta il notebook nella cartella che contiene llm_anatomy/, da dovunque
# sia stato aperto: gli script si eseguono da li', e cosi' anche le celle.
import os
import subprocess
import sys
from pathlib import Path

radice = next((p for p in [Path.cwd(), *Path.cwd().parents]
               if (p / "llm_anatomy").is_dir()), None)

if radice is None:
    # Il notebook e' stato aperto da solo, per esempio su Colab: il codice, il
    # testo e il modello addestrato si scaricano una volta, qui accanto.
    REPO, RAMO, CARTELLA = "fcollova/llm-anatomy", "main", ""
    copia = Path("anatomia-di-un-llm")
    if not copia.exists():
        esito = subprocess.run(
            ["git", "clone", "-q", "--depth", "1", "--branch", RAMO,
             f"https://github.com/{REPO}.git", str(copia)], capture_output=True)
        if esito.returncode != 0:
            raise SystemExit("Non riesco a scaricare il codice da github.com/"
                             + REPO + ": " + esito.stderr.decode()[-300:])
    radice = copia / CARTELLA

os.chdir(radice)
sys.path.insert(0, str(Path.cwd()))

In [2]:
import torch
import torch.nn as nn

from llm_anatomy.utils import prepara, stampa_shape, titolo, verifica

prepara()

d_in, d_mid, d_out, B = 6, 16, 4, 128

L1 = nn.Linear(d_in, d_mid)
L2 = nn.Linear(d_mid, d_out)
X = torch.randn(B, d_in)

[ambiente] torch 2.13.0+cpu  device=cpu  threads=8  seed=1337


In [3]:
titolo("Due strati, uno dopo l'altro")
h = L1(X)
y = L2(h)
stampa_shape("X", X)
stampa_shape("dopo L1", h)
stampa_shape("dopo L2", y)


Due strati, uno dopo l'altro
----------------------------
  X                            (128, 6)               float32
  dopo L1                      (128, 16)              float32
  dopo L2                      (128, 4)               float32


In [4]:
titolo("Una matrice sola fa lo stesso lavoro")
W = L2.weight @ L1.weight
b = L2.bias + L2.weight @ L1.bias
stampa_shape("W = W2 W1", W)
stampa_shape("b = b2 + W2 b1", b)
y_eq = X @ W.T + b
scarto = (y - y_eq).abs().max().item()
print(f"  scarto massimo fra i due percorsi: {scarto:.3e}")
verifica(torch.allclose(y, y_eq, atol=1e-5),
         "un solo strato riproduce esattamente la coppia di strati")
print(f"  parametri usati: {d_in * d_mid + d_mid + d_mid * d_out + d_out} "
      f"contro {d_in * d_out + d_out} necessari")


Una matrice sola fa lo stesso lavoro
------------------------------------
  W = W2 W1                    (4, 6)                 float32
  b = b2 + W2 b1               (4)                    float32
  scarto massimo fra i due percorsi: 1.788e-07
  [verifica superata] un solo strato riproduce esattamente la coppia di strati
  parametri usati: 180 contro 28 necessari


In [5]:
titolo("Con qualcosa in mezzo, non piu'")
y_nl = L2(torch.relu(L1(X)))
scarto_nl = (y_nl - y_eq).abs().max().item()
print(f"  scarto contro la matrice equivalente: {scarto_nl:.3f}")

# La miglior approssimazione lineare possibile: soluzione ai minimi quadrati
# sull'insieme dei punti. Se la rete con ReLU fosse lineare, il residuo
# sarebbe nullo a meno dell'errore numerico.
X1 = torch.cat([X, torch.ones(B, 1)], dim=1)
soluzione = torch.linalg.lstsq(X1, y_nl).solution
residuo = (X1 @ soluzione - y_nl).pow(2).mean().sqrt().item()
varianza = y_nl.std().item()
print(f"  residuo della migliore approssimazione lineare: {residuo:.4f}")
print(f"  deviazione standard dell'uscita:                {varianza:.4f}")
print(f"  quota di segnale non spiegabile linearmente:    "
      f"{100 * residuo / varianza:.1f}%")
verifica(residuo > 0.05 * varianza,
         "nessuna trasformazione lineare riproduce la rete con ReLU")


Con qualcosa in mezzo, non piu'
-------------------------------
  scarto contro la matrice equivalente: 1.410
  residuo della migliore approssimazione lineare: 0.1042
  deviazione standard dell'uscita:                0.2522
  quota di segnale non spiegabile linearmente:    41.3%
  [verifica superata] nessuna trasformazione lineare riproduce la rete con ReLU


In [6]:
titolo("Controprova")
soluzione2 = torch.linalg.lstsq(X1, y).solution
residuo2 = (X1 @ soluzione2 - y).pow(2).mean().sqrt().item()
print(f"  residuo: {residuo2:.3e}  (zero a meno dell'errore numerico)")
verifica(residuo2 < 1e-4,
         "senza attivazione i minimi quadrati ricostruiscono la rete esatta")


Controprova
-----------
  residuo: 9.344e-08  (zero a meno dell'errore numerico)
  [verifica superata] senza attivazione i minimi quadrati ricostruiscono la rete esatta


## Da provare

sostituisci la ReLU con una funzione lineare: la controprova torna a passare.